In [1]:
from pyspark.sql.functions import col, to_timestamp

# 1. Read Bronze Delta Table
df_bronze = spark.read.table("bronze_match_events")

# 2. Clean, type-cast, and deduplicate
df_silver = df_bronze.select(
    col("match_id"),
    col("round_num").cast("int"),
    to_timestamp(col("timestamp")).alias("event_time"),
    col("team"),
    col("player_id"),
    col("event_type"),
    col("coord_x").cast("double"),
    col("coord_y").cast("double"),
    col("hp_remaining").cast("int"),
    col("current_loadout_value").cast("int"),
    col("opponent_alive_count").cast("int")
).dropDuplicates(["match_id", "event_time", "player_id", "event_type"])

# 3. Write to Silver Delta Table
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_match_events")
print("Silver table generated successfully.")

StatementMeta(, 8ab5bc3b-4362-4a06-9289-8acdad7b1f54, 3, Finished, Available, Finished, False)

Silver table generated successfully.


In [2]:
from pyspark.sql import functions as F

df_silver = spark.read.table("silver_match_events")

# Aggregate round dynamics and calculate rolling combat power
df_gold = df_silver.groupBy("match_id", "round_num", "team").agg(
    F.avg("current_loadout_value").alias("avg_loadout_val"),
    F.sum(F.when(F.col("event_type") == "kill", 1).otherwise(0)).alias("round_kills"),
    F.avg("hp_remaining").alias("team_avg_hp"),
    # Combat Power Index: 60% economy weight + 40% team health weight
    ( (F.avg("current_loadout_value") / 5000.0) * 0.6 + (F.avg("hp_remaining") / 100.0) * 0.4 ).alias("combat_power_index")
)

df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_round_features")
print("Gold feature table created successfully.")

StatementMeta(, 8ab5bc3b-4362-4a06-9289-8acdad7b1f54, 4, Finished, Available, Finished, False)

Gold feature table created successfully.
